In [ ]:
import matplotlib.pyplot as plt
import numpy as np

g = 9.81 #[m/s2]
B_canal = 5 #[m]
n_manning = 0.035
Q_caudal = 25 #[m3/s]
y_inicial = 10 #[m]
x_inicial = 0 #[m]
x_final = -10000 #[m]
paso_x = -1 #[m]

So_1 = 0.001 #[m/m]
So_2 = 0.002 #[m/m]
x_cambio_pendiente = x_final*0.5

def profundidad_critica(Q, B, g):
  termino = (Q**2) / (g * B**2)
  y_c = termino**(1/3)
  return y_c

def area_canal(B, y):
  return B*y

def perimetro_canal(B, y):
  return B + 2*y

def radio_hidraulico(B, y):
  return (area_canal(B, y))/perimetro_canal(B, y)

def ancho_superficial(B,y):
  return B

def profundidad_hidraulica(B, y):
  return area_canal(B, y)/ancho_superficial(B,y)

def funcion_sf(n, Q, B, y):
  primer_termino = n*(Q/area_canal(B,y))
  segundo_termino = radio_hidraulico(B, y)
  return (primer_termino/(segundo_termino)**(2/3))**2

def funcion_froude(Q, B, y):
  primer_termino = Q/area_canal(B,y)
  segundo_termino = g*profundidad_hidraulica(B,y)
  return primer_termino/np.sqrt(segundo_termino)


def funcion_tasa_profundidad_largo(n, Q, B, y, So):
  primer_termino = So - funcion_sf(n, Q, B, y)
  segundo_termino = 1 - (funcion_froude(Q, B, y))**2
  return primer_termino/segundo_termino

print(funcion_tasa_profundidad_largo(n_manning, Q_caudal, B_canal, y_inicial, So_1))
print(funcion_sf(n_manning, Q_caudal, B_canal, y_inicial)>So_1)


def funcion_siguiente_profundidad(n, Q, B, y, So):
  return y + funcion_tasa_profundidad_largo(n, Q, B, y, So)*paso_x

x = x_inicial
y = y_inicial
So = So_1
x_lista = []
y_lista = []

while x >= x_final:
  x_lista.append(x)
  y_lista.append(y)
  y = funcion_siguiente_profundidad(n_manning, Q_caudal, B_canal, y, So)
  x = x + paso_x

  if x > x_cambio_pendiente:
    So = So_1
  else:
    So = So_2

print(So)
print(funcion_froude(Q_caudal, B_canal, y_inicial))
print(funcion_sf(n_manning, Q_caudal, B_canal, y_inicial))
print(profundidad_critica(Q_caudal, B_canal, g))

zb_lista = []

for x in x_lista:
  if x > x_cambio_pendiente:
    zb_lista.append(-So_1 * x)
  else:
    zb_cambio_pendiente = -So_1 * x_cambio_pendiente
    zb_lista.append(-So_2 * x - zb_cambio_pendiente)

wse_lista = [zb + y for zb, y in zip(zb_lista, y_lista)]


plt.figure(figsize=(15, 4))
plt.plot(x_lista, wse_lista, label='Superficie del Agua', color='blue')
plt.plot(x_lista, zb_lista, label='Fondo del Canal (So={})'.format(So), color='black', linewidth=2)
plt.fill_between(x_lista, zb_lista, wse_lista, color='lightblue', alpha=0.6)
plt.legend()
plt.grid()
plt.show()
